In [1]:
# Cellule 1 — Imports
from fastmcp import FastMCP
from datetime import datetime
import json
import asyncio

print("Imports OK !")

Imports OK !


In [2]:
# Cellule 2 — Création du serveur
mcp = FastMCP("Assistant Energie EDF")

@mcp.tool()
def get_current_date() -> str:
    """Retourne la date du jour au format YYYY-MM-DD.
    Utiliser quand l'utilisateur demande des données 
    récentes sans préciser de date."""
    return datetime.now().strftime("%Y-%m-%d")

@mcp.tool()
def get_consommation_electrique(
    date_debut: str,
    date_fin: str,
    region: str = "France"
) -> str:
    """Récupère les données de consommation électrique.
    
    Args:
        date_debut: Date de début au format YYYY-MM-DD
        date_fin: Date de fin au format YYYY-MM-DD
        region: Région française (défaut: France entière)
    
    Returns:
        Données de consommation en JSON
    """
    donnees = {
        "region": region,
        "periode": f"{date_debut} → {date_fin}",
        "consommation_mwh": 45230,
        "variation_vs_annee_precedente": "-3.2%",
        "pic_journalier": "19h00",
        "source": "Données simulées - API RTE"
    }
    return json.dumps(donnees, ensure_ascii=False, indent=2)

@mcp.tool()
def get_mix_energetique(date: str) -> str:
    """Récupère le mix de production énergétique pour une date donnée.
    
    Args:
        date: Date au format YYYY-MM-DD
    
    Returns:
        Répartition de la production par source en JSON
    """
    mix = {
        "date": date,
        "production_totale_mwh": 52100,
        "repartition": {
            "nucleaire": "71.2%",
            "hydraulique": "12.4%",
            "eolien": "8.1%",
            "solaire": "4.3%",
            "thermique": "3.8%",
            "autres": "0.2%"
        },
        "taux_co2_gco2_kwh": 38,
        "source": "Données simulées - API RTE"
    }
    return json.dumps(mix, ensure_ascii=False, indent=2)

@mcp.tool()
def get_prix_electricite(
    date_debut: str,
    date_fin: str
) -> str:
    """Récupère les prix de l'électricité sur le marché spot.
    
    Args:
        date_debut: Date de début YYYY-MM-DD
        date_fin: Date de fin YYYY-MM-DD
    
    Returns:
        Prix moyens en euros par MWh en JSON
    """
    prix = {
        "periode": f"{date_debut} → {date_fin}",
        "prix_moyen_eur_mwh": 87.50,
        "prix_min_eur_mwh": 42.10,
        "prix_max_eur_mwh": 156.80,
        "tendance": "hausse",
        "source": "Données simulées - EPEX SPOT"
    }
    return json.dumps(prix, ensure_ascii=False, indent=2)

print("Serveur MCP créé avec 4 outils !")
print("Outils : get_current_date, get_consommation_electrique, get_mix_energetique, get_prix_electricite")

Serveur MCP créé avec 4 outils !
Outils : get_current_date, get_consommation_electrique, get_mix_energetique, get_prix_electricite


In [3]:
# Cellule 3 — Test des outils directement
print("Test 1 — Date du jour :")
print(get_current_date())

print("\nTest 2 — Consommation électrique :")
print(get_consommation_electrique(
    date_debut="2024-01-01",
    date_fin="2024-01-31",
    region="Île-de-France"
))

print("\nTest 3 — Mix énergétique :")
print(get_mix_energetique(date="2024-01-15"))

print("\nTest 4 — Prix électricité :")
print(get_prix_electricite(
    date_debut="2024-01-01",
    date_fin="2024-01-31"
))

Test 1 — Date du jour :
2026-04-02

Test 2 — Consommation électrique :
{
  "region": "Île-de-France",
  "periode": "2024-01-01 → 2024-01-31",
  "consommation_mwh": 45230,
  "variation_vs_annee_precedente": "-3.2%",
  "pic_journalier": "19h00",
  "source": "Données simulées - API RTE"
}

Test 3 — Mix énergétique :
{
  "date": "2024-01-15",
  "production_totale_mwh": 52100,
  "repartition": {
    "nucleaire": "71.2%",
    "hydraulique": "12.4%",
    "eolien": "8.1%",
    "solaire": "4.3%",
    "thermique": "3.8%",
    "autres": "0.2%"
  },
  "taux_co2_gco2_kwh": 38,
  "source": "Données simulées - API RTE"
}

Test 4 — Prix électricité :
{
  "periode": "2024-01-01 → 2024-01-31",
  "prix_moyen_eur_mwh": 87.5,
  "prix_min_eur_mwh": 42.1,
  "prix_max_eur_mwh": 156.8,
  "tendance": "hausse",
  "source": "Données simulées - EPEX SPOT"
}


In [5]:
# Cellule 4 — Test via le client MCP
from fastmcp import Client

async def test_via_client_mcp():
    print("Test via le protocole MCP\n")
    print("=" * 40)
    
    async with Client(mcp) as client:
        
        # Lister les outils disponibles
        tools = await client.list_tools()
        print(f"Outils disponibles : {len(tools)}")
        for tool in tools:
            print(f"  - {tool.name}")
        
        print("\n")
        
        # Appeler un outil via MCP
        print("Appel MCP - get_consommation_electrique :")
        result = await client.call_tool(
            "get_consommation_electrique",
            {
                "date_debut": "2024-01-01",
                "date_fin": "2024-01-31",
                "region": "Paris"
            }
        )
        print(result)
        
        print("\nAppel MCP - get_mix_energetique :")
        result = await client.call_tool(
            "get_mix_energetique",
            {"date": "2024-01-15"}
        )
        print(result)

# Lancement du test
await test_via_client_mcp()

Test via le protocole MCP

Outils disponibles : 4
  - get_current_date
  - get_consommation_electrique
  - get_mix_energetique
  - get_prix_electricite


Appel MCP - get_consommation_electrique :
CallToolResult(content=[TextContent(type='text', text='{\n  "region": "Paris",\n  "periode": "2024-01-01 → 2024-01-31",\n  "consommation_mwh": 45230,\n  "variation_vs_annee_precedente": "-3.2%",\n  "pic_journalier": "19h00",\n  "source": "Données simulées - API RTE"\n}', annotations=None, meta=None)], structured_content={'result': '{\n  "region": "Paris",\n  "periode": "2024-01-01 → 2024-01-31",\n  "consommation_mwh": 45230,\n  "variation_vs_annee_precedente": "-3.2%",\n  "pic_journalier": "19h00",\n  "source": "Données simulées - API RTE"\n}'}, meta={'fastmcp': {'wrap_result': True}}, data='{\n  "region": "Paris",\n  "periode": "2024-01-01 → 2024-01-31",\n  "consommation_mwh": 45230,\n  "variation_vs_annee_precedente": "-3.2%",\n  "pic_journalier": "19h00",\n  "source": "Données simulées - AP

In [7]:
# Cellule 5 — Configuration LangSmith
import os
from dotenv import load_dotenv

# Charger le .env avec le chemin exact
load_dotenv("/Users/aminatadiallo/assistant-edf/assistant-edf/.env")

# Vérification
langchain_key = os.getenv("LANGCHAIN_API_KEY")
langchain_project = os.getenv("LANGCHAIN_PROJECT")
tracing = os.getenv("LANGCHAIN_TRACING_V2")

print(f"LangSmith Key : {langchain_key[:20]}..." if langchain_key else "ERREUR : cle manquante !")
print(f"Projet : {langchain_project}")
print(f"Tracing active : {tracing}")

LangSmith Key : lsv2_pt_a010cacf4151...
Projet : assistant-edf
Tracing active : true


In [10]:
# Cellule 6 — Test LangSmith avec un vrai appel LLM
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Initialiser le LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

# Faire un appel simple
print("Envoi d'une question au LLM...")
response = llm.invoke([
    HumanMessage(content="Qu'est-ce que le mix energetique en France ? Reponds en 2 phrases.")
])

print("\nReponse du LLM :")
print(response.content)
print("\nVa sur smith.langchain.com pour voir la trace !")

Envoi d'une question au LLM...

Reponse du LLM :
Le mix énergétique en France désigne la répartition des différentes sources d'énergie utilisées pour la production d'électricité, comprenant notamment le nucléaire, les énergies renouvelables et les énergies fossiles. Actuellement, le nucléaire représente la majeure partie du mix énergétique en France, suivi par les énergies renouvelables en pleine croissance.

Va sur smith.langchain.com pour voir la trace !
